In [1]:
import scipy.io
import numpy as np
import os
from scipy.io.wavfile import write

# 1. 경로 설정
mat_dir = "C:/Users/enjoy\Downloads/archive (2)/raw" 
wav_dir = "C:/Users/enjoy\Downloads/archive (2)/raw/converted_wav"
os.makedirs(wav_dir, exist_ok=True)

# 2. 샘플링 주파수 설정
fs = 12000  # Hz

# 3. 변환 함수
def convert_mat_to_wav(mat_path, wav_path):
    mat = scipy.io.loadmat(mat_path)

    # 시계열 key 자동 탐색
    for key in mat.keys():
        if isinstance(mat[key], np.ndarray) and mat[key].ndim == 2 and mat[key].shape[1] == 1:
            signal = mat[key].flatten()
            break
    else:
        print(f" 시계열 데이터 못 찾음: {mat_path}")
        return

    # 정규화 후 int16 변환
    signal = signal / np.max(np.abs(signal))  # -1 ~ 1
    signal = (signal * 32767).astype(np.int16)

    # 저장
    write(wav_path, fs, signal)
    print(f" 저장됨: {wav_path}")

# 4. 폴더 내 .mat 전부 변환
for file in os.listdir(mat_dir):
    if file.endswith(".mat"):
        mat_path = os.path.join(mat_dir, file)
        wav_filename = file.replace(".mat", ".wav")
        wav_path = os.path.join(wav_dir, wav_filename)

        convert_mat_to_wav(mat_path, wav_path)

 저장됨: C:/Users/enjoy\Downloads/archive (2)/raw/converted_wav\B007_1_123.wav
 저장됨: C:/Users/enjoy\Downloads/archive (2)/raw/converted_wav\B014_1_190.wav
 저장됨: C:/Users/enjoy\Downloads/archive (2)/raw/converted_wav\B021_1_227.wav
 저장됨: C:/Users/enjoy\Downloads/archive (2)/raw/converted_wav\IR007_1_110.wav
 저장됨: C:/Users/enjoy\Downloads/archive (2)/raw/converted_wav\IR014_1_175.wav
 저장됨: C:/Users/enjoy\Downloads/archive (2)/raw/converted_wav\IR021_1_214.wav
 저장됨: C:/Users/enjoy\Downloads/archive (2)/raw/converted_wav\OR007_6_1_136.wav
 저장됨: C:/Users/enjoy\Downloads/archive (2)/raw/converted_wav\OR014_6_1_202.wav
 저장됨: C:/Users/enjoy\Downloads/archive (2)/raw/converted_wav\OR021_6_1_239.wav
 저장됨: C:/Users/enjoy\Downloads/archive (2)/raw/converted_wav\Time_Normal_1_098.wav


In [ ]:
C:/Users/enjoy\Downloads/archive (2)/raw/converted_wav"

In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.io import wavfile
from scipy.fft import rfft, rfftfreq

# ✅ 1. 특징 추출 함수
def extract_features_from_wav(wav_path):
    sr, data = wavfile.read(wav_path)
    data = data.astype(np.float32)
    data = (data - np.mean(data)) / (np.max(np.abs(data)) + 1e-6)

    # FFT 특성
    fft_vals = np.abs(rfft(data))
    fft_freqs = rfftfreq(len(data), 1 / sr)
    fft_peak_idx = np.argmax(fft_vals)
    fft_peak_freq = fft_freqs[fft_peak_idx]
    fft_peak_mag = fft_vals[fft_peak_idx]

    # 통계 + 스펙트럼 특성
    features = {
        'max': np.max(data),
        'min': np.min(data),
        'mean': np.mean(data),
        'std': np.std(data),
        'rms': np.sqrt(np.mean(np.square(data))),
        'skewness': pd.Series(data).skew(),
        'kurtosis': pd.Series(data).kurt(),
        'crest_factor': np.max(np.abs(data)) / (np.sqrt(np.mean(np.square(data))) + 1e-6),
        'form_factor': np.sqrt(np.mean(np.square(data))) / (np.mean(np.abs(data)) + 1e-6),
        'abs_mean': np.mean(np.abs(data)),
        'iqr': np.percentile(data, 75) - np.percentile(data, 25),
        'zcr': ((data[:-1] * data[1:]) < 0).sum(),
        'fft_peak_freq': fft_peak_freq,
        'fft_peak_mag': fft_peak_mag
    }

    return pd.DataFrame([features])

# ✅ 2. 라벨 단순화
def simplify_fault_from_filename(filename):
    if filename.startswith('B'):
        return 'Ball'
    elif filename.startswith('IR'):
        return 'IR'
    elif filename.startswith('OR'):
        return 'OR'
    else:
        return 'Normal'

# ✅ 3. 데이터 경로 설정
wav_dir = "C:/Users/enjoy/Downloads/archive (2)/raw/converted_wav"
out_csv_path = "C:/Users/enjoy\Downloads/archive (2)/raw/static/feature.csv"

features_list = []

# ✅ 4. 반복 처리
for fname in tqdm(os.listdir(wav_dir)):
    if not fname.endswith(".wav"):
        continue
    fpath = os.path.join(wav_dir, fname)

    try:
        features = extract_features_from_wav(fpath)
        features['label'] = simplify_fault_from_filename(fname)
        features['filename'] = fname
        features_list.append(features)
    except Exception as e:
        print(f"❌ {fname} 처리 실패: {e}")

# ✅ 5. CSV 저장
df_features = pd.concat(features_list, ignore_index=True)
df_features.to_csv(out_csv_path, index=False)
print(f"✅ 저장 완료: {out_csv_path}")

100%|██████████| 10/10 [00:00<00:00, 11.46it/s]

✅ 저장 완료: C:/Users/enjoy\Downloads/archive (2)/raw/static/feature.csv
